In [0]:
import os, sys
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), "src"))
from lh import common

dbutils.widgets.dropdown("env", "dev", ["dev", "test", "prod"])
cfg = common.load_config(dbutils.widgets.get("env"))
t = f"{cfg['catalog']}.gold.dim_date"
results = []

def check(name, ok, detail=""):
    results.append(bool(ok))
    print("PASS" if ok else "FAIL", name, detail)

def row(day):
    return spark.sql(f"SELECT * FROM {t} WHERE date = DATE'{day}'").first()

In [0]:
s = spark.sql(f"""SELECT count(*) AS n, count(DISTINCT date_key) AS k, min(date) AS a, max(date) AS b,
                         datediff(max(date), min(date)) + 1 AS span FROM {t}""").first()
check("1095 days, 2025-01-01 to 2027-12-31", s["n"] == 1095 and str(s["a"]) == "2025-01-01" and str(s["b"]) == "2027-12-31", s["n"])
check("date_key is unique", s["k"] == s["n"])
check("no missing day", s["span"] == s["n"])

a, b, c = row("2026-03-31"), row("2026-04-01"), row("2026-10-04")
check("31 Mar 2026 closes FY2025-26 quarter 4", (a["fiscal_year"], a["fiscal_quarter"], a["is_month_end"]) == ("FY2025-26", 4, True))
check("1 Apr 2026 opens FY2026-27 quarter 1", (b["fiscal_year"], b["fiscal_quarter"]) == ("FY2026-27", 1))
check("4 Oct 2026 is a Sunday, key 20261004",
      (c["date_key"], c["day_name"], c["day_of_week"], c["is_weekend"]) == (20261004, "Sunday", 7, True))

print(f"\nt17_gold_dim_date: {sum(results)} of {len(results)} checks passed ->", "PASS" if all(results) else "FAIL")
if not all(results):
    raise RuntimeError("t17_gold_dim_date failed")